# المختبر الختامي لليوم الثاني — بناء نموذج تصنيف كامل
**الهدف:** الانتقال من ملف منافذ إلى توقعات توقف العملاء، ثم مقارنة Baseline وLogistic Regression وDecision Tree مقارنة أولية. نفّذ الخلايا بالترتيب واقرأ الشرح قبل كل خطوة.

In [ ]:
import io, pandas as pd
from google.colab import files
uploaded = files.upload()
file_name = next(iter(uploaded))
df = pd.read_csv(io.BytesIO(uploaded[file_name]))
print(df.shape)
display(df.head(3))

## 1) نفصل الهدف عن الخصائص
نحفظ `y` أولًا، ثم نستبعد المعرّف والهدف وأعمدة المستقبل من `X`.

In [ ]:
y = df['churned_30d'].copy()
drop_cols = ['customer_id','churned_30d','refund_issued','support_ticket_after_snapshot','next_month_orders','signup_date','snapshot_date']
X = df.drop(columns=drop_cols, errors='ignore')
print('X:', X.shape, 'y:', y.shape)
print(y.value_counts(normalize=True).round(3))

## 2) التقسيم الطبقي
يحافظ `stratify=y` على نسبة 0 و1 تقريبًا في التدريب والاختبار.

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print(X_train.shape, X_test.shape)
print(y_train.mean().round(3), y_test.mean().round(3))

## 3) نبني المعالجة المسبقة (Preprocessing)
يتعلم التعويض والترميز والتحجيم من بيانات التدريب فقط، ثم يطبق القواعد نفسها على الاختبار.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric = X.select_dtypes(include='number').columns.tolist()
categorical = X.select_dtypes(exclude='number').columns.tolist()

num_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler())
])
cat_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('encode', OneHotEncoder(handle_unknown='ignore'))
])
prep = ColumnTransformer([
    ('num', num_pipe, numeric),
    ('cat', cat_pipe, categorical)
])
print('أصبحت وصفة المعالجة جاهزة ✅')

## 4) نبني ثلاثة Pipelines
نستخدم المعالجة نفسها، ونغيّر النموذج فقط: Baseline بسيط، ثم Logistic Regression، ثم Decision Tree محدودة العمق.

In [ ]:
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

baseline = Pipeline([
    ('prep', clone(prep)),
    ('model', DummyClassifier(strategy='most_frequent'))
])
logistic = Pipeline([
    ('prep', clone(prep)),
    ('model', LogisticRegression(max_iter=1000))
])
tree = Pipeline([
    ('prep', clone(prep)),
    ('model', DecisionTreeClassifier(max_depth=4, random_state=42))
])

baseline.fit(X_train, y_train)
logistic.fit(X_train, y_train)
tree.fit(X_train, y_train)
print('تم تدريب النماذج الثلاثة ✅')

## 5) مقارنة أولية فقط
نحسب نسبة النتائج الصحيحة لفحص أن الرحلة تعمل. لا نختار النموذج الأفضل اليوم؛ لأن Accuracy قد تكون مضللة عندما تكون إحدى الفئات قليلة، وسنتعلم المقاييس المناسبة في اليوم الثالث.

In [ ]:
from sklearn.metrics import accuracy_score

models = {
    'Baseline': baseline,
    'Logistic Regression': logistic,
    'Decision Tree': tree,
}

for name, fitted_model in models.items():
    predictions = fitted_model.predict(X_test)
    score = accuracy_score(y_test, predictions)
    print(name, 'accuracy =', round(score, 3))

## 6) نقرأ احتمالات Logistic Regression
الاحتمال ليس حقيقة مؤكدة ولا قرارًا نهائيًا. نعرضه الآن، ونؤجل اختيار العتبة المناسبة إلى اليوم الثالث.

In [ ]:
probability = logistic.predict_proba(X_test)[:, 1]
result = pd.DataFrame({
    'الحقيقة': y_test.iloc[:10].values,
    'احتمال_التوقف': probability[:10].round(3),
})
display(result)

## التسليم
1. أحجام التدريب والاختبار. 2. نتائج المقارنة الأولية للنماذج الثلاثة. 3. عشرة احتمالات من Logistic Regression. 4. جملة تشرح لماذا لا تكفي Accuracy لاختيار الفائز النهائي.